# Basic Investment Model — GMM Validation

This notebook validates the Generalized Method of Moments (GMM) implementation using the basic investment model with convex adjustment costs ($\psi_1 > 0$). GMM is applicable because the Euler equation provides closed-form moment conditions that can be evaluated directly from observables $(\pi, k, I)$ without solving the model.

**Two-step estimation procedure:**

- **Stage 1:** Minimize $Q(\beta)$ with $W = I$ using `dual_annealing` (global search) + Powell refinement.
- **Stage 2:** Warm-start from $\hat{\beta}_1$ with $W = \hat{\Omega}^{-1}$, local Powell only.

**Tables produced:**

1. Moment vector — $g(\hat{\beta})$ at the final estimate (should be near zero)
2. Parameter estimates — true, estimate, SE, t-test, J-test
3. Monte Carlo summary — bias, RMSE, SD, SE across $J$ replications

**Note:** The fake-real panel is generated by solving the model once at the true parameters via PFI, then simulating. The GMM estimator treats this panel as observed data and never solves the model. Each $Q(\beta)$ evaluation is arithmetic on the data, making GMM orders of magnitude faster than SMM.

In [ ]:
from pathlib import Path
import os
import sys
import time
import warnings
from dataclasses import asdict

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

warnings.filterwarnings(
    'ignore',
    message=r'.*does not produce the same series as CPU implementation.*',
)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from src.v2.estimation import (
    GMMMonteCarloConfig,
    GMMRunConfig,
    solve_gmm,
    validate_gmm,
)
from src.v2.evaluation import (
    prepare_evaluation_run,
    save_manifest_sections,
    save_summary_rows,
)
from src.v2.environments.basic_investment import (
    BasicInvestmentEnv,
    BasicInvestmentSMMSolverConfig,
    EconomicParams,
    ShockParams,
)
from src.v2.solvers.config import GridConfig, PFIConfig
from src.v2.utils.seeding import fold_in_seed

np.set_printoptions(precision=6, suppress=True)

MOMENT_LABELS = {
    'euler_const': 'Euler x 1',
    'euler_ik_lag': 'Euler x I/k lag',
    'euler_pik_lag': 'Euler x pi/k lag',
    'shock_const': 'Shock x 1',
    'shock_lnz_lag': 'Shock x ln z lag',
    'variance_const': 'Var x 1',
}
PARAM_LABELS = {
    'alpha': 'alpha',
    'psi1': 'psi1',
    'rho': 'rho',
    'sigma_epsilon': 'sigma',
}

## Config

The fake-real panel is generated once by solving the model at true parameters via PFI. The GMM estimator then treats $(\pi, k, I)$ as observed data. Each optimizer evaluation is microseconds (arithmetic on the panel), so wall time is dominated by the one-time PFI solve for panel generation.

**Solver precision matters.** Unlike SMM (which matches summary statistics), GMM exploits the Euler equation at every observation. PFI approximation error in the policy enters the Euler residual as systematic bias. Use a dense PFI grid to keep this error below the sampling noise floor. The `SOLVER_CONFIG` below controls grid density explicitly.

In [ ]:
EXPERIMENT_NAME = 'basic_investment_gmm_validation'
RESULTS_ROOT   = str(REPO_ROOT / 'outputs' / 'notebooks')

SAVE_RUN = True
RUN_TAG  = 'gmm-validation-basic-investment'

PROFILE = 'MC_VALIDATION'  # Options: 'SMOKE', 'MC_VALIDATION'

PROFILES = {
    'SMOKE': {
        'description': 'Quick smoke test with moderate panel and grid.',
        'panel_n_firms': 50,
        'panel_horizon': 25,
        'panel_burn_in': 175,
        'pfi_grid': {'exo_sizes': [50], 'endo_sizes': [100], 'action_sizes': [100]},
        'gmm': {
            'optimizer_name': 'dual_annealing',
            'optimizer_maxiter': 30,
            'master_seed': (20, 26),
        },
        'mc': {'n_replications': 3},
    },
    'MC_VALIDATION': {
        'description': 'Full Monte Carlo validation with dense grid.',
        'panel_n_firms': 100,
        'panel_horizon': 25,
        'panel_burn_in': 275,
        'pfi_grid': {'exo_sizes': [50], 'endo_sizes': [100], 'action_sizes': [100]},
        'gmm': {
            'optimizer_name': 'dual_annealing',
            'optimizer_maxiter': 100,
            'master_seed': (20, 26),
        },
        'mc': {'n_replications': 20},
    },
}

assert PROFILE in PROFILES, f'Unknown profile: {PROFILE!r}'
profile = PROFILES[PROFILE]

# Fixed at calibrated values -- not estimated by GMM.
CALIBRATED = dict(
    interest_rate=0.04,
    depreciation_rate=0.15,
)

# True values of the 4 structural parameters estimated by GMM.
TRUE_PARAMS = dict(
    production_elasticity=0.6,
    cost_convex=0.3,
    rho=0.70,
    sigma=0.15,
)

# Optimizer search region.
BETA_BOUNDS = (
    (0.01, 0.95),   # alpha
    (0.0, 1.0),     # psi1
    (0.0, 0.95),    # rho
    (0.01, 0.99),   # sigma
)

ECON_PARAMS = EconomicParams(
    **CALIBRATED,
    production_elasticity=TRUE_PARAMS['production_elasticity'],
    cost_convex=TRUE_PARAMS['cost_convex'],
)
SHOCK_PARAMS = ShockParams(mu=0.0, rho=TRUE_PARAMS['rho'], sigma=TRUE_PARAMS['sigma'])

# -- PFI solver config for fake-real panel generation --
# GMM exploits the Euler equation at every (i, t), so PFI policy
# approximation error enters the residuals directly.  Use a dense grid
# to keep this error below the sampling noise floor.
# The solve happens once per panel (not inside the GMM optimizer loop),
# so a dense grid is affordable.
SOLVER_CONFIG = BasicInvestmentSMMSolverConfig(
    method='PFI',
    pfi_config=PFIConfig(
        grid=GridConfig(**profile['pfi_grid']),
        max_iter=200,
        eval_steps=400,
    ),
)

GMM_RUN_CONFIG = GMMRunConfig(**profile['gmm'])
MC_CONFIG = GMMMonteCarloConfig(**profile['mc'])

RUN = prepare_evaluation_run(
    EXPERIMENT_NAME,
    save_run=SAVE_RUN,
    run_tag=RUN_TAG,
    results_root=RESULTS_ROOT,
)

print(f'Profile: {PROFILE} \u2014 {profile["description"]}')
print(f'Output: {RUN["run_dir"]}')
print(f'Panel: N={profile["panel_n_firms"]}, T={profile["panel_horizon"]}, burn_in={profile["panel_burn_in"]}')
print(f'PFI grid: exo={SOLVER_CONFIG.pfi_config.grid.exo_sizes}, '
      f'endo={SOLVER_CONFIG.pfi_config.grid.endo_sizes}, '
      f'action={SOLVER_CONFIG.pfi_config.grid.action_sizes}')
print(f'GMM: optimizer={GMM_RUN_CONFIG.optimizer_name}, maxiter={GMM_RUN_CONFIG.optimizer_maxiter}')
print(f'MC:  n_replications={MC_CONFIG.n_replications}')

save_manifest_sections(
    RUN,
    setup={'profile': PROFILE, 'description': profile['description']},
    calibrated=CALIBRATED,
    true_params=TRUE_PARAMS,
    solver_config={
        'method': SOLVER_CONFIG.method,
        'pfi_grid_exo': SOLVER_CONFIG.pfi_config.grid.exo_sizes,
        'pfi_grid_endo': SOLVER_CONFIG.pfi_config.grid.endo_sizes,
        'pfi_grid_action': SOLVER_CONFIG.pfi_config.grid.action_sizes,
        'pfi_max_iter': SOLVER_CONFIG.pfi_config.max_iter,
    },
    gmm_run=asdict(GMM_RUN_CONFIG),
    monte_carlo=asdict(MC_CONFIG),
)

Profile: MC_VALIDATION — Full Monte Carlo validation with dense grid.
Output: /Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_gmm_validation/gmm-validation-basic-investment
Panel: N=100, T=25, burn_in=275
PFI grid: exo=[50], endo=[100], action=[100]
GMM: optimizer=dual_annealing, maxiter=100
MC:  n_replications=20


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_gmm_validation/gmm-validation-basic-investment/manifest.json')

## Environment and Panel Data

Solve the model once at true parameters via PFI, then simulate the fake-real observed panel $(\pi, k, I)$. The GMM estimator treats this panel as data and never re-solves the model.

In [3]:
env = BasicInvestmentEnv(econ_params=ECON_PARAMS, shock_params=SHOCK_PARAMS)
beta_true = env.gmm_true_beta()

# Generate fake-real panel (one-time PFI solve + simulation).
panel_seed = fold_in_seed(GMM_RUN_CONFIG.master_seed, 'gmm', 'panel')
t0 = time.perf_counter()
panel = env.simulate_gmm_panel(
    seed=panel_seed,
    n_firms=profile['panel_n_firms'],
    horizon=profile['panel_horizon'],
    burn_in=profile['panel_burn_in'],
    solver_config=SOLVER_CONFIG,
)
panel_wall = time.perf_counter() - t0

spec = env.make_gmm_spec(panel, bounds=BETA_BOUNDS, solver_config=SOLVER_CONFIG)

# Parameter overview.
beta_table = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': beta_true,
        'initial_guess': spec.initial_guess,
        'lower': [b[0] for b in spec.bounds],
        'upper': [b[1] for b in spec.bounds],
    }
)
display(beta_table)
print(f'Panel: {panel.n_firms} firms x {panel.horizon} periods, '
      f'NT_eff = {spec.n_observations} (generated in {panel_wall:.1f}s)')
print(f'Moments: {[MOMENT_LABELS[m] for m in spec.moment_names]}')

# Sanity check: g(beta_true) should be near zero.
g_true = spec.compute_moments(beta_true)
print(f'g(beta_true) max |g| = {np.max(np.abs(g_true)):.6f}')

PFI converged: 6 policy updates


,parameter,true,initial_guess,lower,upper
0,alpha,0.60,0.480,0.01,0.95
1,psi1,0.10,0.500,0.00,1.00
2,rho,0.70,0.475,0.00,0.95
3,sigma,0.15,0.500,0.01,0.99


Panel: 100 firms x 25 periods, NT_eff = 2300 (generated in 128.0s)
Moments: ['Euler x 1', 'Euler x I/k lag', 'Euler x pi/k lag', 'Shock x 1', 'Shock x ln z lag', 'Var x 1']
g(beta_true) max |g| = 0.003398


## Two-Step GMM

Run the full two-step GMM. Stage 1 uses $W = I$ (global search). Stage 2 warm-starts from $\hat{\beta}_1$ with $W = \hat{\Omega}^{-1}$ (local Powell only).

**Table 1** reports the moment vector $g(\hat{\beta})$ — all 6 conditions should be near zero.
**Table 2** reports parameter estimates with standard errors and t-tests.

In [4]:
# Estimate GMM
start = time.perf_counter()
gmm_result = solve_gmm(spec, config=GMM_RUN_CONFIG)
gmm_wall_time = time.perf_counter() - start

In [5]:
# GMM results summary

# --- Table 1: Moment Vector ---
moment_table = pd.DataFrame(
    {
        'moment': [MOMENT_LABELS[m] for m in spec.moment_names],
        'g(beta_hat)': gmm_result.stage2.moment_vector,
    }
)
print('Table 1. Moment Conditions at beta_hat')
display(moment_table)

# --- Table 2: Parameter Estimates ---
beta_hat = gmm_result.beta_hat
se = gmm_result.standard_errors
t_stat = (beta_hat - beta_true) / np.where(se > 0, se, np.nan)
p_value = 2.0 * (1.0 - norm.cdf(np.abs(t_stat)))

param_est = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': beta_true,
        'estimate': beta_hat,
        'se': se,
        't_stat: βₖ = β_true': t_stat,
        'p_value: βₖ = β_true': p_value,
    }
)
print('Table 2. Parameter Estimates')
display(param_est)

# --- Estimation info ---
est_info = [
    {'key': 'j_statistic', 'value': gmm_result.j_statistic},
    {'key': 'j_p_value', 'value': gmm_result.j_p_value},
    {'key': 'j_df', 'value': gmm_result.j_df},
    {'key': 'stage1_nfev', 'value': gmm_result.stage1.optimizer_nfev},
    {'key': 'stage2_nfev', 'value': gmm_result.stage2.optimizer_nfev},
    {'key': 'wall_time_sec', 'value': round(gmm_wall_time, 2)},
]
print(f'\nJ = {gmm_result.j_statistic:.4f}, p = {gmm_result.j_p_value:.4f}, df = {gmm_result.j_df}')
print(f'Stage 1: nfev = {gmm_result.stage1.optimizer_nfev}')
print(f'Stage 2: nfev = {gmm_result.stage2.optimizer_nfev}')
print(f'Wall time: {gmm_wall_time:.1f}s')

# --- Export ---
save_summary_rows(RUN, moment_table.to_dict('records'), filename='moment_fit.csv')
save_summary_rows(RUN, param_est.to_dict('records'), filename='parameter_estimates.csv')
save_summary_rows(RUN, est_info, filename='estimation_info.csv')

Table 1. Moment Conditions at beta_hat


,moment,g(beta_hat)
0,Euler x 1,-9.233198e-04
1,Euler x I/k lag,-1.697195e-04
2,Euler x pi/k lag,-2.656827e-04
3,Shock x 1,-3.548369e-03
4,Shock x ln z lag,6.873188e-04
5,Var x 1,-3.709787e-07


Table 2. Parameter Estimates


,parameter,true,estimate,se,t_stat: βₖ = β_true,p_value: βₖ = β_true
0,alpha,0.60,0.599924,0.000556,-0.137417,0.890701
1,psi1,0.10,0.086108,0.017590,-0.789775,0.429659
2,rho,0.70,0.681728,0.015425,-1.184531,0.236203
3,sigma,0.15,0.146583,0.002140,-1.596807,0.110309



J = 4.1141, p = 0.1278, df = 2
Stage 1: nfev = 975
Stage 2: nfev = 246
Wall time: 0.2s


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_gmm_validation/gmm-validation-basic-investment/estimation_info.csv')

## Monte Carlo Validation

Repeat the full two-step GMM on $J$ independent fake-real panels. Each replication re-solves the model once (PFI) to generate a new panel, then runs GMM. **Table 3** reports bias, RMSE, SD, and average SE across replications.

In [6]:
start = time.perf_counter()
mc_result = validate_gmm(spec, beta_true, run_config=GMM_RUN_CONFIG, mc_config=MC_CONFIG)
mc_wall_time = time.perf_counter() - start

# --- Table 3: Monte Carlo Summary ---
mc_summary = pd.DataFrame(
    {
        'parameter': [PARAM_LABELS[p] for p in spec.parameter_names],
        'true': mc_result.summary.beta_true,
        'mean_estimate': mc_result.summary.mean_beta_hat,
        'bias': mc_result.summary.bias,
        'rmse': mc_result.summary.rmse,
        'sd': mc_result.summary.sd,
        'avg_se': mc_result.summary.mean_standard_error,
    }
)
print('Table 3. Monte Carlo Summary')
display(mc_summary)

# --- MC info ---
mc_info = [
    {'key': 'n_replications', 'value': MC_CONFIG.n_replications},
    {'key': 'optimizer_maxiter', 'value': GMM_RUN_CONFIG.optimizer_maxiter},
    {'key': 'j_test_reject_rate', 'value': round(float(mc_result.summary.j_test_size), 4)},
    {'key': 'wall_time_sec', 'value': round(mc_wall_time, 2)},
]
print(f'\nJ-test reject rate (5%): {mc_result.summary.j_test_size:.2f}')
print(f'MC wall time: {mc_wall_time:.1f}s ({MC_CONFIG.n_replications} replications)')

# --- Export ---
save_summary_rows(RUN, mc_summary.to_dict('records'), filename='mc_summary.csv')
save_summary_rows(RUN, mc_info, filename='mc_info.csv')

PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 7 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 7 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
PFI converged: 6 policy updates
Table 3. Monte Carlo Summary


,parameter,true,mean_estimate,bias,rmse,sd,avg_se
0,alpha,0.60,0.601461,0.001461,0.002110,0.001562,0.000810
1,psi1,0.10,0.134513,0.034513,0.057563,0.047265,0.033169
2,rho,0.70,0.685551,-0.014449,0.030289,0.027312,0.018767
3,sigma,0.15,0.150618,0.000618,0.002840,0.002844,0.002244



J-test reject rate (5%): 0.20
MC wall time: 2586.4s (20 replications)


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/basic_investment_gmm_validation/gmm-validation-basic-investment/mc_info.csv')

## Conclusion

This notebook validates the GMM implementation on the basic investment model with convex adjustment costs. Unlike SMM, each optimizer evaluation is arithmetic on the data (no model solve), making GMM orders of magnitude faster. The one-time PFI solve per Monte Carlo replication is the dominant cost, but this is only for validation on simulated dataset. We do not need to solve the model when applying the GMM to actual real-world data. 

For models without closed-form Euler equations (e.g., risky debt with default), SMM is required.